# 91 · A discovery screen along the PT: level, location and gradient survival

A condition effect along the axis, Δ(x), is split into its **level** (the mean over the axis: does the whole PT
change?), its **location** (the rest: where along the PT does it change?) and **gradient survival** (how much of a
gene's healthy gradient remains). Two filters then attribute level changes: are they **kidney-wide** (the same change
in DCT and thin-limb units) and do they look like **time of day, fasting or sex** in public mouse kidney data? Species
differences use gradient survival only, because probe panels make levels incomparable. Every gene and pathway is
scored against relabeled specimens and expression-matched random sets; the ranked lists are the candidates for
follow-up.

**Protocol.** `docs/results/pt-positional-decomposition.md` (frozen before any fit, commit 8b739ae).

**What bounds the reading.** Two control and two IR male mice, one male human donor (two cortex sections). Every q
is a ranking device. IR and control can differ in harvest time, surgery and feeding, which no relabeling removes.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, '1')
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from IPython.display import display

EXPORT_AKI = results_root / 'package_pseudospace' / 'aki' / 'pt_units.h5ad'
T71 = results_root / 'pt_pathways_package_pseudospace' / 'tables'
T80 = results_root / 'pt_known_injury_programs' / 'tables'
T90 = results_root / 'pt_continuous_axis_experiments' / 'tables'
STATE = data_root / 'external' / 'kidney_physiological_state'
XIONG = data_root / 'external' / 'xiong2023_sex_bias'
output = results_root / 'pt_positional_decomposition'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)

FROZEN_SHA = {  # sha256 recorded when the protocol was frozen
    T90 / 'd_end_contrasts.csv.gz': 'b4a5646f71f998fa241ef8fb05f30f9a2151a331ebfeb5ed58ac0ea80d9effe1',
    T90 / 'a_gene_classes.csv.gz': '48d3394334fd18c7fb7c6b7728bfec49a75303b74be49831b47ffffe6bdf156b',
    T71 / 'gene_sets_aki_vs_control.json': '034112512c80d4236497c5f7677829e99c67eafcd33ed645429b16cd93a0cdc4',
    T71 / 'gene_sets_human_vs_mouse.json': '00f3804d1c7649813ac474b3044897c5b6c1aaf3e31aaf4e7c15be891d90019f',
    T80 / 'programs_members.csv': '28a615591ba9b24cf78d640bf0ab6978bd96dd09bb751877b0696e26238c4b3c',
    STATE / 'GSE277302_Normalized_counts.csv.gz': 'ceba9ac458b9e725757b2f15567fe66e6f174f436a90752757e669fa79a8515c',
    STATE / 'GSE267280_RNA_Seq_counts.txt.gz': 'bf1607f9528be2f847bc29dcdc2bf34f83e212378b4e6af8d755518fa8841d0a',
    XIONG / 'PT-S1_fWTvsmWT.csv': '4764f77435e1b232e2e686770776c16b3c9f93a085eee1b52d9914b116f00eee',
    XIONG / 'PT-S2_fWTvsmWT.csv': '59e9103e6210a43d6569166d8d3e40b3ec1e089e3caadcbe4244bd8a23c220a6',
    XIONG / 'PT-S3_fWTvsmWT.csv': '966286c7ae61b2f78db9b6f3aa4fe3d9c942d67527b0ca309f96ed8ded9cd9cd',
}
REQUIRED = [EXPORT_AKI, STATE / 'GSE277302_series_matrix.txt.gz', STATE / 'GSM9744085_CTRL_ZT4_01_features.tsv.gz',
            *FROZEN_SHA]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70, 71, 80 and 90 first; see data/README.md)')

NOTEBOOK_LOGIC_VERSION = '91.positional_decomposition.1'
FINGERPRINT_AKI = 'bb4a80c1b484'
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))
SEED = 91
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
AKI4 = CONTROLS + IRS
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
LABELS = ('PT-S1', 'PT-S2', 'PT-S3')
NON_PT = ('DCT', 'Thin_Limb')
# Protocol constants
DETECTION, REF_DET, PROGRAM_DET = .02, .05, .10
LEVEL, LOCATION, R_REP, RELABEL = .5, .4, .5, .5
ZON_R, ZON_RANGE, STRETCH_EFFECT = .5, 1., .5
N_PERM, MIN_SET, MIN_ZON, Q = 2000, 10, 5, .05
RELABEL_A = .67
KIDNEY_WIDE, PT_ONLY, MIN_NONPT_UNITS = .5, .25, 150
TOD_LFC, TOD_P, FAST_LFC, FAST_CPM = .5, .01, 1., 10.
DIP_LO, DIP_HI, DIP_FRAC, DIP_AGREE = .35, 1.95, .1, .3
GRADED, LATE = .5, 1.8
STRETCHES = {'S1→S2': ((0., .8), (1.2, 1.8)), 'S2→S3': ((1.2, 1.8), (2.1, 2.5))}

MM = 1 / 25.4
INK, LIGHT, GREY = '#0b0b0b', '#d9d8d4', '#9a9893'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'IR2A2': '#B2182B', 'IR2A4': '#EF8A62'}
CALL_COLOR = {'level': '#0072B2', 'location': '#D55E00', 'both': '#CC79A7'}
CLASS_COLOR = {'discrete': '#7f7f7f', 'discrete + within': '#b5a642', 'gradient': '#0072B2', 'switch': '#009E73',
               'bump': '#D55E00', 'trough': '#CC79A7', 'flat': LIGHT}
SURV_COLOR = {'reversed': '#B2182B', 'erased': '#EF8A62', 'attenuated': '#FDDBC7', 'kept': '#92C5DE',
              'amplified': '#2166AC'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 200)
GUARDS, READING = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def panel(ax, letter):
    ax.text(-.02, 1.02, letter, transform=ax.transAxes, fontsize=8, fontweight='bold', ha='right', va='bottom')


def seg_lines(ax, cuts=(1, 2), vertical=True):
    for b in cuts:
        (ax.axvline if vertical else ax.axhline)(b, color=LIGHT, lw=.6, zorder=0)


def short(s, n=46):
    name = s.split('::', 1)[-1]
    for tail in (' R-HSA-', ' WP'):
        if tail in name:
            name = name.split(tail)[0]
    return name if len(name) <= n else name[:n - 1] + '…'


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Inputs and guards (not cached)

In [ ]:
import anndata as ad
import gzip
from scipy import sparse, stats
from scipy.stats import spearmanr
from joblib import Parallel, delayed
from statsmodels.stats.multitest import multipletests

from pseudospace import continuous_axis as ca
from pseudospace import positional_decomposition as pdc
from pseudospace import positional_programs as pp
from pseudospace import species_calls as sc
from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.package_inputs import read_pt_units, support_positions
from pseudospace.stage_cache import cached_frame, cached_payload, digest

for path, expected in FROZEN_SHA.items():
    assert sha256(path) == expected, f'G1: {path.name} changed since the protocol was frozen'
GUARDS.append(f'G1: {len(FROZEN_SHA)} frozen inputs match their protocol sha256')

AKI = read_pt_units(EXPORT_AKI, 'aki')
assert AKI.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT_AKI
obs = AKI.obs
spec_all = obs['sample'].astype(str).to_numpy()
seg_all = obs.segment_class.astype(str).to_numpy()
MASK, pos = support_positions(obs.pseudospace.to_numpy(float), spec_all, seg_all)
SPEC, SEG = spec_all[MASK], seg_all[MASK]
IDENT = obs.primary_identity.astype(str).to_numpy()[MASK]
EXT = obs['curve_end_extension'].to_numpy(float)[MASK]
TR = logistic_transitions(pos, SEG, SPEC)
for n in CONTROLS:
    t = TR[TR.specimen.eq(n)].sort_values('transition').position.to_numpy()
    assert len(t) == 2 and t[0] < t[1], f'G2: {n} transitions out of order'
CUTS = sc.mean_transitions(TR, CONTROLS)
assert np.allclose(np.round(CUTS, 3), (.339, .657)), f'G2: cuts {CUTS} differ from 90'
UNITS = sc.to_segment_units(pos, CUTS)
GUARDS.append(f'G2: AKI control cuts {CUTS[0]:.3f}, {CUTS[1]:.3f} equal 90\'s')
X_ALL = sparse.csr_matrix(AKI.X, dtype=float)[np.flatnonzero(MASK)]
GENES_ALL = pd.Index(AKI.var_names.astype(str))
NOT_MT = ~GENES_ALL.str.startswith('mt-')
LIB_ALL = np.asarray(X_ALL[:, np.flatnonzero(NOT_MT)].sum(axis=1)).ravel()
units_by = pd.crosstab(IDENT, SPEC)
assert all(units_by.loc[i, n] >= MIN_NONPT_UNITS for i in NON_PT for n in AKI4), units_by
GUARDS.append('G3: DCT and Thin_Limb hold ≥ 150 units in every mouse ' + json.dumps(units_by.loc[list(NON_PT)].to_dict()))

PT = IDENT == 'Proximal_Tubule'
X = X_ALL[np.flatnonzero(PT)]
spec, units, seg, lib, ext = SPEC[PT], UNITS[PT], SEG[PT], LIB_ALL[PT], EXT[PT]
assert set(np.unique(seg)) <= set(LABELS)
DET = np.array([np.asarray((X[spec == n] > 0).mean(axis=0)).ravel() for n in AKI4])
dc_, di_ = DET[:2], DET[2:]
SHARED = (DET >= DETECTION).all(axis=0) & NOT_MT
INDUCED = (dc_ < DETECTION).all(axis=0) & (di_ >= DETECTION).all(axis=0) & NOT_MT
LOST = (di_ < DETECTION).all(axis=0) & (dc_ >= DETECTION).all(axis=0) & NOT_MT
SH = np.flatnonzero(SHARED)
GENES = np.array(GENES_ALL[SH])
REF = (DET[:, SH] >= REF_DET).all(axis=0)
BIN, CENTRES = pp.fixed_bins(units)
NB = len(CENTRES)
RATES = pp.bin_profiles(X[:, SH], lib, BIN, NB, spec, AKI4, centre=False)
VALID = np.all([np.isfinite(RATES[n][0]) for n in AKI4], axis=0)
for n in AKI4:
    RATES[n][:, ~VALID] = np.nan
RATE_C = np.asarray(X[np.isin(spec, CONTROLS)][:, SH].sum(axis=0)).ravel() / lib[np.isin(spec, CONTROLS)].sum()
LOG_CP = np.log2(RATE_C * 1e4 + 1e-6)
COUNTS = {n: np.bincount(BIN[(spec == n) & (BIN >= 0)], minlength=NB).astype(float) for n in AKI4}
print('\n'.join('· ' + g for g in GUARDS))
print(f'PT units {pd.Series(spec).value_counts().to_dict()}; genes: shared {SHARED.sum():,} (reference {REF.sum():,}), '
      f'induced {INDUCED.sum():,}, lost {LOST.sum():,}; valid bins {CENTRES[VALID].min():.2f}–{CENTRES[VALID].max():.2f}')

code_key = digest([NOTEBOOK_LOGIC_VERSION, Path(pdc.__file__).read_text(), Path(ca.__file__).read_text(),
                   Path(pp.__file__).read_text()])
SECONDS = {}


def cached_table(stage, compute, *, params=None, inputs=None):
    t = time.perf_counter()
    out = cached_frame(stage, compute, root=cache, params=params, inputs=inputs, code=code_key)
    SECONDS[stage] = time.perf_counter() - t
    return out

## 3 · I1 · Level and location

Δ_K(x) = mean case − mean reference binned log2 rate, centred per bin by the median reference gene. L is its mean over
the valid bins, P = Δ − L, M = RMS of P. Calls need the four IR × control pairs to agree and both relabelings to be
at most half the condition.

In [ ]:
def delta(case, ref):
    d = np.mean([RATES[n] for n in case], axis=0) - np.mean([RATES[n] for n in ref], axis=0)
    return pp.centre_by_median(d, REF)


CONTRASTS = {'condition': (IRS, CONTROLS),
             'R1': (['IR2A2', 'Ctrl1A2'], ['IR2A4', 'Ctrl1A4']),
             'R2': (['IR2A2', 'Ctrl1A4'], ['IR2A4', 'Ctrl1A2']),
             'IR2A2 vs controls': (['IR2A2'], CONTROLS), 'IR2A4 vs controls': (['IR2A4'], CONTROLS),
             'R1 case 1': (['IR2A2'], ['IR2A4', 'Ctrl1A4']), 'R1 case 2': (['Ctrl1A2'], ['IR2A4', 'Ctrl1A4']),
             'R2 case 1': (['IR2A2'], ['IR2A4', 'Ctrl1A2']), 'R2 case 2': (['Ctrl1A4'], ['IR2A4', 'Ctrl1A2'])}
PAIRS = [(i, c) for i in IRS for c in CONTROLS]
for i, c in PAIRS:
    CONTRASTS[f'{i}-{c}'] = ([i], [c])
DELTA = {k: delta(*v) for k, v in CONTRASTS.items()}
DEC = {k: pdc.level_location(d) for k, d in DELTA.items()}   # k -> (L, P, M)
L, P, M = DEC['condition']
R_P = pdc.row_corr(DEC['IR2A2 vs controls'][1], DEC['IR2A4 vs controls'][1])
pair_sign = np.all([np.sign(DEC[f'{i}-{c}'][0]) == np.sign(L) for i, c in PAIRS], axis=0)
LEVEL_CALL = (np.abs(L) >= LEVEL) & pair_sign & np.all([np.abs(DEC[r][0]) <= RELABEL * np.abs(L) for r in ('R1', 'R2')], axis=0)
LOC_CALL = (M >= LOCATION) & (R_P >= R_REP) & np.all([DEC[r][2] <= RELABEL * M for r in ('R1', 'R2')], axis=0)
G = pd.DataFrame({'gene': GENES, 'reference': REF, 'log2 CP10k (controls)': LOG_CP, 'L': L, 'M': M, 'r_P': R_P,
                  **{f'L {i}-{c}': DEC[f'{i}-{c}'][0] for i, c in PAIRS},
                  'L R1': DEC['R1'][0], 'L R2': DEC['R2'][0], 'M R1': DEC['R1'][2], 'M R2': DEC['R2'][2],
                  'level change': LEVEL_CALL, 'location change': LOC_CALL,
                  'peak': CENTRES[np.nanargmax(np.where(np.isfinite(P), np.abs(P), -1), axis=1)],
                  'Δ late': np.nanmean(DELTA['condition'][:, VALID & (CENTRES >= LATE)], axis=1)})
G['call'] = np.select([LEVEL_CALL & LOC_CALL, LEVEL_CALL, LOC_CALL], ['both', 'level', 'location'], 'none')
I1_COUNTS = pd.crosstab(G['call'], np.where(G.L > 0, 'up', 'down'))
display(I1_COUNTS)
I1_SPEC = {}
for r, (case, ref) in {'R1': CONTRASTS['R1'], 'R2': CONTRASTS['R2']}.items():
    Lr, Pr, Mr = DEC[r]
    others = [k for k in ('condition', 'R1', 'R2') if k != r]
    I1_SPEC[r] = {'level': int(((np.abs(Lr) >= LEVEL) & np.all([np.abs(DEC[o][0]) <= RELABEL * np.abs(Lr) for o in others], axis=0)).sum()),
                  'location': int(((Mr >= LOCATION) & np.all([DEC[o][2] <= RELABEL * Mr for o in others], axis=0)).sum())}
I1_SPEC['condition'] = {'level': int(((np.abs(L) >= LEVEL) & np.all([np.abs(DEC[o][0]) <= RELABEL * np.abs(L) for o in ('R1', 'R2')], axis=0)).sum()),
                        'location': int(((M >= LOCATION) & np.all([DEC[o][2] <= RELABEL * M for o in ('R1', 'R2')], axis=0)).sum())}
READING.append('I1 post hoc: the level and location rules without the pair and replication parts, applied to each '
               'relabeling as the contrast (the other two splits as its relabelings), as a specificity check.')
print('I1 calls without pair/replication parts (condition and relabelings as contrast):', I1_SPEC)

### Injury shape (90's ladder on Δ)

Replicates: the two IR mice's Δ against the mean of the controls; weights the harmonic mean of the IR mouse's and the
controls' units per bin; label fractions from the controls. The same on each relabeling (its two case specimens).

In [ ]:
def harmonic(a, b):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where((a > 0) & (b > 0), 2 / (1 / a + 1 / b), 0.)


def ladder_for(name, keys, case_names, ref_names):
    ref_units = np.isin(spec, ref_names)
    frac = ca.label_fractions(BIN[ref_units], seg[ref_units], NB, LABELS)
    ref_n = np.sum([COUNTS[n] for n in ref_names], axis=0)
    profs = [DELTA[k] for k in keys]
    w = [harmonic(COUNTS[c], ref_n) for c in case_names]

    def run():
        t = ca.fit_ladder(profs, w, CENTRES, [frac, frac])
        t.insert(0, 'gene', GENES)
        return t
    return cached_table(f'ladder_{name}', run, inputs={'p': profs, 'w': w, 'f': frac})


LADDERS = {'condition': ladder_for('condition', ['IR2A2 vs controls', 'IR2A4 vs controls'], IRS, CONTROLS),
           'R1': ladder_for('R1', ['R1 case 1', 'R1 case 2'], ['IR2A2', 'Ctrl1A2'], ['IR2A4', 'Ctrl1A4']),
           'R2': ladder_for('R2', ['R2 case 1', 'R2 case 2'], ['IR2A2', 'Ctrl1A4'], ['IR2A4', 'Ctrl1A2'])}
SHAPE = {k: ca.ladder_class(t.drop(columns='gene')) for k, t in LADDERS.items()}
for col in ('class', 'location', 'continuous vs discrete', 'dBIC continuous'):
    G[f'injury {col}'] = SHAPE['condition'][col].to_numpy()
CLASS_ORDER = ['discrete', 'discrete + within', 'gradient', 'switch', 'bump', 'trough']
SHAPE_NULL = pd.DataFrame({k: SHAPE[k]['class'].value_counts() for k in SHAPE}).reindex(CLASS_ORDER + ['flat']).fillna(0).astype(int)
display(SHAPE_NULL)
SHAPE_LOC = G[G['location change']].groupby(['injury class', 'injury location']).size().unstack(fill_value=0)
display(SHAPE_LOC)
display(G[G['location change']]['injury continuous vs discrete'].value_counts())

## 4 · I2 · Gradient survival

Deviation profiles per mouse (90's). Zonated in controls: the two control profiles correlate at r ≥ 0.5 and their
mean spans ≥ 1 log2. A_{i,t} = Σ d_i·d_t / Σ d_h·d_t with template t and held-out control h; the gene's A is the
median over the four (i, t).

In [ ]:
DEV = {}
for n in AKI4:
    y = RATES[n] - np.nanmean(RATES[n], axis=1, keepdims=True)
    DEV[n] = y - np.nanmedian(y[REF], axis=0)
r_ctrl = pdc.row_corr(DEV['Ctrl1A2'], DEV['Ctrl1A4'])
ZONATED = (r_ctrl >= ZON_R) & (np.nanmax((DEV['Ctrl1A2'] + DEV['Ctrl1A4']) / 2, axis=1)
                               - np.nanmin((DEV['Ctrl1A2'] + DEV['Ctrl1A4']) / 2, axis=1) >= ZON_RANGE)
SURV_SPECS = {'condition': (IRS, CONTROLS), 'R1': (['IR2A2', 'Ctrl1A2'], ['IR2A4', 'Ctrl1A4']),
              'R2': (['IR2A2', 'Ctrl1A4'], ['IR2A4', 'Ctrl1A2'])}
SURV = {k: pdc.cross_fitted_survival([DEV[n] for n in case], [DEV[n] for n in ref]) for k, (case, ref) in SURV_SPECS.items()}
G['zonated'] = ZONATED
G['r controls'] = r_ctrl
for k in SURV:
    G[f'A {k}'] = SURV[k][0]
G['survival'] = np.where(ZONATED, pdc.survival_class(G['A condition']), '')
SURV_TAB = pd.DataFrame({k: {'whole PT': pdc.pooled_ratio(SURV[k][1], SURV[k][2], ZONATED)} for k in SURV})
for s, (a, b) in STRETCHES.items():
    ia = VALID & (CENTRES >= a[0]) & (CENTRES < a[1])
    ib = VALID & (CENTRES >= b[0]) & (CENTRES < b[1])
    cst = {n: (np.nanmean(DEV[n][:, ib], axis=1) - np.nanmean(DEV[n][:, ia], axis=1))[:, None] for n in AKI4}
    zs = (np.abs(cst['Ctrl1A2'][:, 0]) >= STRETCH_EFFECT) & (np.abs(cst['Ctrl1A4'][:, 0]) >= STRETCH_EFFECT) & \
         (np.sign(cst['Ctrl1A2'][:, 0]) == np.sign(cst['Ctrl1A4'][:, 0]))
    for k, (case, ref) in SURV_SPECS.items():
        _, num, den = pdc.cross_fitted_survival([cst[n] for n in case], [cst[n] for n in ref])
        SURV_TAB.loc[f'{s} ({int(zs.sum())} genes)', k] = pdc.pooled_ratio(num, den, zs)
SURV_TAB.loc['zonated genes'] = int(ZONATED.sum())
display(SURV_TAB.round(3))
SURV_COUNTS = G.loc[ZONATED, 'survival'].value_counts().reindex(list(pdc.SURVIVAL_CLASSES)).fillna(0).astype(int)
SURV_NULL = pd.DataFrame({k: pd.Series(pdc.survival_class(G.loc[ZONATED, f'A {k}'])).value_counts() for k in SURV}) \
    .reindex(list(pdc.SURVIVAL_CLASSES)).fillna(0).astype(int)
display(SURV_NULL)
display(G[ZONATED].sort_values('A condition')[['gene', 'A condition', 'A R1', 'A R2', 'r controls', 'L', 'survival']].head(25))

## 5 · I3 · Kidney-wide filter

DCT and thin-limb units of the same mice: pseudobulk log2 CPM, IR mean − control mean, centred by the median
reference gene of that identity. Kidney-wide when both identities share L's sign with |FC| ≥ 0.5 |L|; PT-only when
both are < 0.25 |L|.

In [ ]:
FC_NONPT = {}
for ident in NON_PT:
    rows = np.flatnonzero(IDENT == ident)
    Xn = X_ALL[rows]
    det_n = np.array([np.asarray((Xn[SPEC[rows] == n] > 0).mean(axis=0)).ravel() for n in AKI4])
    ref_n = (det_n >= REF_DET).all(axis=0)[SH]
    lc = pdc.pseudobulk_log_cpm(Xn[:, SH], LIB_ALL[rows], SPEC[rows], AKI4)
    fc = lc[2:].mean(axis=0) - lc[:2].mean(axis=0)
    FC_NONPT[ident] = fc - np.median(fc[ref_n])
    G[f'FC {ident}'] = FC_NONPT[ident]
same = np.all([(np.sign(FC_NONPT[i]) == np.sign(L)) & (np.abs(FC_NONPT[i]) >= KIDNEY_WIDE * np.abs(L)) for i in NON_PT], axis=0)
pt_only = np.all([np.abs(FC_NONPT[i]) < PT_ONLY * np.abs(L) for i in NON_PT], axis=0)
G['kidney-wide'] = np.select([same, pt_only], ['kidney-wide', 'PT-only'], 'mixed')
KW = pd.crosstab(G.loc[LEVEL_CALL, 'kidney-wide'], np.where(G.loc[LEVEL_CALL, 'L'] > 0, 'up', 'down'))
display(KW)
print('Spearman of PT L with the non-PT FC (all shared genes):',
      {i: round(float(spearmanr(L, FC_NONPT[i], nan_policy='omit')[0]), 3) for i in NON_PT})

## 6 · I4 · Induced and lost programs along the axis (unit level)

Induced: off in both controls (< 2%), on in ≥ 10% of units of both IR mice. Lost: the reverse. Per unit, the share
of the program's genes detected; per bin and mouse, the mean share. A replicated interior dip needs both IR mice.

In [ ]:
IND = np.flatnonzero(INDUCED & (di_ >= PROGRAM_DET).all(axis=0))
LOS = np.flatnonzero(LOST & (dc_ >= PROGRAM_DET).all(axis=0))
SHARE = {'induced': np.asarray((X[:, IND] > 0).mean(axis=1)).ravel(), 'lost': np.asarray((X[:, LOS] > 0).mean(axis=1)).ravel()}


def program_profile(share, keep):
    out = {}
    for n in AKI4:
        r = keep & (spec == n) & (BIN >= 0)
        cnt = np.bincount(BIN[r], minlength=NB)
        s = np.bincount(BIN[r], weights=share[r], minlength=NB)
        with np.errstate(invalid='ignore', divide='ignore'):
            out[n] = np.where(cnt >= pp.MIN_UNITS, s / cnt, np.nan)
    return out


PROG = {(p, sens): program_profile(SHARE[p], np.ones(len(spec), bool) if sens == 'all' else ext <= 0)
        for p in SHARE for sens in ('all', 'on curve')}
DIPS = []
for sens in ('all', 'on curve'):
    for n in IRS:
        dip, where, excess = pdc.interior_dip(PROG[('induced', sens)][n], CENTRES, lo=DIP_LO, hi=DIP_HI, frac=DIP_FRAC)
        DIPS.append({'units': sens, 'mouse': n, 'dip': dip, 'position': where, 'excess / range': excess})
DIPS = pd.DataFrame(DIPS)
DIP_REP = {sens: bool(d.dip.all() and np.ptp(d.position) <= DIP_AGREE + 1e-9) for sens, d in DIPS.groupby('units')}
display(DIPS)
print(f'I4: induced program {len(IND)} genes, lost program {len(LOS)} genes; replicated interior dip: {DIP_REP}')
IND_GENES = pd.DataFrame({'gene': GENES_ALL[IND], 'detection IR2A2': di_[0, IND], 'detection IR2A4': di_[1, IND],
                          'detection Ctrl1A2': dc_[0, IND], 'detection Ctrl1A4': dc_[1, IND]}).sort_values('detection IR2A4', ascending=False)
LOS_GENES = pd.DataFrame({'gene': GENES_ALL[LOS], 'detection Ctrl1A2': dc_[0, LOS], 'detection Ctrl1A4': dc_[1, LOS],
                          'detection IR2A2': di_[0, LOS], 'detection IR2A4': di_[1, LOS]}).sort_values('detection Ctrl1A2', ascending=False)
print('induced (top):', IND_GENES.gene.head(20).tolist())
print('lost (top):', LOS_GENES.gene.head(20).tolist())
READING.append('I4 post hoc: each induced gene\'s detection in early S1 (bins < 0.55) against S3 (bins ≥ 1.95) per IR '
               'mouse; early when both mice show ≥ 2-fold, late when both show ≤ 0.5-fold. Seen in figure 91b, panel e.')
EARLY_BINS, LATE_BINS = VALID & (CENTRES < .55), VALID & (CENTRES >= 1.95)
for n in IRS:
    r = spec == n
    e_ = r & np.isin(BIN, np.flatnonzero(EARLY_BINS))
    l_ = r & np.isin(BIN, np.flatnonzero(LATE_BINS))
    de = np.asarray((X[e_][:, IND] > 0).mean(axis=0)).ravel()
    dl_ = np.asarray((X[l_][:, IND] > 0).mean(axis=0)).ravel()
    IND_GENES.loc[IND_GENES.gene.isin(GENES_ALL[IND]), f'log2 early/late {n}'] = pd.Series(
        np.log2((de + .01) / (dl_ + .01)), index=GENES_ALL[IND]).reindex(IND_GENES.gene).to_numpy()
ratio_cols = [f'log2 early/late {n}' for n in IRS]
IND_GENES['zone'] = np.select([(IND_GENES[ratio_cols] >= 1).all(axis=1), (IND_GENES[ratio_cols] <= -1).all(axis=1)],
                              ['early S1', 'S3'], 'both')
display(IND_GENES.sort_values(ratio_cols[0], ascending=False).round(3))
print('I4 post hoc zones:', IND_GENES.zone.value_counts().to_dict())
ir_units = np.isin(spec, IRS)
TOP_IND = pd.Series(np.arange(len(GENES_ALL)), index=GENES_ALL)[IND_GENES.assign(m=IND_GENES[ratio_cols].mean(axis=1)).sort_values('m', ascending=False).gene].to_numpy()
TOP_DET = np.vstack([[np.mean(X[ir_units & (BIN == b), g].toarray() > 0) if (ir_units & (BIN == b)).sum() >= pp.MIN_UNITS else np.nan
                      for b in range(NB)] for g in TOP_IND])

## 7 · I5 · Sets: the discovery output

Library sets (71's AKI sets) and external programs (80's published injury programs; time of day, fasting and sex
from public mouse kidney data). Statistics: mean L; the members' mean location profile and its size; set gradient
survival. Null: 2,000 random sets matched member-for-member by expression decile; z, normal p, BH per statistic.

In [ ]:
features = pd.read_csv(STATE / 'GSM9744085_CTRL_ZT4_01_features.tsv.gz', sep='\t', header=None, names=['id', 'symbol', 'type'])
ens_to_symbol = dict(zip(features.id, features.symbol))
with gzip.open(STATE / 'GSE277302_series_matrix.txt.gz', 'rt') as handle:
    lines = [l.rstrip('\n').split('\t') for l in handle if l.startswith('!Sample_title') or l.startswith('!Sample_characteristics_ch1')]
titles = [x.strip('"') for x in lines[0][1:]]
chars = {row[1].strip('"').split(':')[0]: [x.strip('"').split(': ')[1] for x in row[1:]] for row in lines[1:]}
meta = pd.DataFrame({'sex': chars['Sex'], 'period': chars['treatment']}, index=titles)
diurnal = pd.read_csv(STATE / 'GSE277302_Normalized_counts.csv.gz', index_col=0)
diurnal['symbol'] = diurnal.EnsemblID.map(ens_to_symbol)
diurnal = np.log2(diurnal.dropna(subset=['symbol']).groupby('symbol')[titles].sum() + 1)
act = meta.index[meta.sex.eq('male') & meta.period.eq('active')]       # ZT18
ina = meta.index[meta.sex.eq('male') & meta.period.eq('inactive')]     # ZT6
TOD = pd.DataFrame({'lfc': diurnal[act].mean(axis=1) - diurnal[ina].mean(axis=1),
                    'p': stats.ttest_ind(diurnal[act], diurnal[ina], axis=1, equal_var=False).pvalue}, index=diurnal.index)
fast = pd.read_csv(STATE / 'GSE267280_RNA_Seq_counts.txt.gz', sep='\t')
fast['symbol'] = fast.Gene_id.str.split('.').str[0].map(ens_to_symbol)
fast = fast.dropna(subset=['symbol']).groupby('symbol')[['Fed_Kidney', 'Fasting_Kidney']].sum()
cpm = fast / fast.sum() * 1e6
FAST = pd.DataFrame({'lfc': np.log2(cpm.Fasting_Kidney + 1) - np.log2(cpm.Fed_Kidney + 1), 'cpm': cpm.mean(axis=1)})
xi = pd.concat([pd.read_csv(XIONG / f'PT-{s}_fWTvsmWT.csv').assign(segment=s) for s in ('S1', 'S2', 'S3')])
SEX = xi.groupby('symbol').log2FC.mean()   # female − male
sig = xi[xi.significant.astype(str).str.upper().eq('TRUE')]
EXTERNAL = {
    'time of day: ZT18-high': TOD.index[(TOD.lfc >= TOD_LFC) & (TOD.p <= TOD_P)].tolist(),
    'time of day: ZT6-high': TOD.index[(TOD.lfc <= -TOD_LFC) & (TOD.p <= TOD_P)].tolist(),
    'fasting: up': FAST.index[(FAST.lfc >= FAST_LFC) & (FAST.cpm >= FAST_CPM)].tolist(),
    'fasting: down': FAST.index[(FAST.lfc <= -FAST_LFC) & (FAST.cpm >= FAST_CPM)].tolist(),
    'sex: male-high (Xiong 2023)': sorted(set(sig.symbol[sig.Direction.eq('DOWN')])),
    'sex: female-high (Xiong 2023)': sorted(set(sig.symbol[sig.Direction.eq('UP')])),
}
prog80 = pd.read_csv(T80 / 'programs_members.csv')
for pid, block in prog80.groupby('program_id', sort=False):
    EXTERNAL[f'injury: {pid}'] = block.gene.astype(str).tolist()
print({k: len(v) for k, v in EXTERNAL.items()})

SETS = json.loads((T71 / 'gene_sets_aki_vs_control.json').read_text())
gidx = pd.Series(np.arange(len(GENES)), index=GENES)
ZIDX = np.flatnonzero(ZONATED)
zpos = pd.Series(np.arange(len(ZIDX)), index=ZIDX)


def members_of(genes):
    m = gidx.reindex(list(dict.fromkeys(genes))).dropna().astype(int).to_numpy()
    return np.sort(m)


STRATA = pd.qcut(LOG_CP, 10, labels=False, duplicates='drop')
STRATA_Z = pd.qcut(LOG_CP[ZIDX], 10, labels=False, duplicates='drop')
rng = np.random.default_rng(SEED)
PERMS = pp.strata_permutations(STRATA, N_PERM, rng)
PERMS_Z = pp.strata_permutations(STRATA_Z, N_PERM, rng)
P_COND = DEC['condition'][1]
P_IR = [DEC['IR2A2 vs controls'][1], DEC['IR2A4 vs controls'][1]]
NUM_Z, DEN_Z = SURV['condition'][1][ZIDX], SURV['condition'][2][ZIDX]


def rms(v):
    return float(np.sqrt(np.nanmean(v ** 2)))


def set_row(name, m):
    out = {'set': name, 'members': len(m)}
    out['L'], out['z L'], out['p L'] = pdc.set_mean_z(L, m, PERMS)
    for i, c in PAIRS:
        out[f'L {i}-{c}'] = float(DEC[f'{i}-{c}'][0][m].mean())
    out['L R1'], out['L R2'] = float(DEC['R1'][0][m].mean()), float(DEC['R2'][0][m].mean())
    prof, out['M'], out['z M'], out['p M'] = pdc.set_profile_z(P_COND, m, PERMS)
    out['peak'] = float(CENTRES[np.nanargmax(np.abs(prof))])
    out['peak sign'] = float(np.sign(prof[np.nanargmax(np.abs(prof))]))
    a, b = (np.nanmean(p[m], axis=0) for p in P_IR)
    ok = np.isfinite(a) & np.isfinite(b)
    out['r IR mice'] = float(np.corrcoef(a[ok], b[ok])[0, 1])
    out['M R1'], out['M R2'] = rms(np.nanmean(DEC['R1'][1][m], axis=0)), rms(np.nanmean(DEC['R2'][1][m], axis=0))
    for i in NON_PT:
        out[f'FC {i}'] = float(FC_NONPT[i][m].mean())
    mz = zpos.reindex(m).dropna().astype(int).to_numpy()
    out['zonated members'] = len(mz)
    if len(mz) >= MIN_ZON:
        out['A'], out['z A'], out['p A'] = pdc.set_ratio_z(NUM_Z, DEN_Z, mz, PERMS_Z)
        out['A R1'] = pdc.pooled_ratio(SURV['R1'][1][ZIDX], SURV['R1'][2][ZIDX], mz)
        out['A R2'] = pdc.pooled_ratio(SURV['R2'][1][ZIDX], SURV['R2'][2][ZIDX], mz)
    return out, prof


def set_tables(sets):
    rows = Parallel(n_jobs=N_JOBS, prefer='threads')(delayed(set_row)(s, m) for s, m in sets.items())
    t = pd.DataFrame([r for r, _ in rows])
    profiles = np.vstack([p for _, p in rows])
    return t, profiles


A_GENOME = pdc.pooled_ratio(SURV['condition'][1], SURV['condition'][2], ZONATED)


def calls(t):
    t = t.copy()
    for s in ('L', 'M', 'A'):
        ok = t[f'p {s}'].notna()
        t.loc[ok, f'q {s}'] = multipletests(t.loc[ok, f'p {s}'], method='fdr_bh')[1]
    pair_ok = np.all([np.sign(t[f'L {i}-{c}']) == np.sign(t.L) for i, c in PAIRS], axis=0)
    t['level set'] = (t['q L'] <= Q) & pair_ok & (t['L R1'].abs() <= RELABEL * t.L.abs()) & (t['L R2'].abs() <= RELABEL * t.L.abs())
    t['location set'] = (t['q M'] <= Q) & (t['r IR mice'] >= R_REP) & (t['M R1'] <= RELABEL * t.M) & (t['M R2'] <= RELABEL * t.M)
    t['gradient set'] = (t['q A'] <= Q) & (t['A R1'] >= RELABEL_A) & (t['A R2'] >= RELABEL_A)
    t['gradient'] = np.where(t['gradient set'], np.where(t.A < A_GENOME, 'erased', 'kept'), '')
    t['name'] = [short(s) for s in t.set]
    return t


LIB_M = {s: members_of(g) for s, g in SETS.items()}
LIB_M = {s: m for s, m in LIB_M.items() if len(m) >= MIN_SET}
EXT_M = {s: members_of(g) for s, g in EXTERNAL.items()}
EXT_M = {s: m for s, m in EXT_M.items() if len(m) >= MIN_SET}
SET_INPUTS = {'L': L, 'P': P_COND, 'perms': PERMS, 'perms_z': PERMS_Z, 'num': NUM_Z, 'den': DEN_Z}


def lib_compute():
    t, prof = set_tables(LIB_M)
    return pd.concat([t, pd.DataFrame(prof, columns=[f'P {c:.2f}' for c in CENTRES])], axis=1)


def ext_compute():
    t, prof = set_tables(EXT_M)
    return pd.concat([t, pd.DataFrame(prof, columns=[f'P {c:.2f}' for c in CENTRES])], axis=1)


LIB = calls(cached_table('i5_library_sets', lib_compute, params={'n_perm': N_PERM, 'seed': SEED},
                         inputs={**SET_INPUTS, 'sets': {s: m for s, m in LIB_M.items()}}))
EXT = calls(cached_table('i5_external_programs', ext_compute, params={'n_perm': N_PERM, 'seed': SEED},
                         inputs={**SET_INPUTS, 'sets': {s: m for s, m in EXT_M.items()}}))
GROUPS = pp.redundancy_groups({s: set(LIB_M[s]) for s in LIB.set})
LIB['group'] = LIB.set.map(GROUPS)
PCOLS = [f'P {c:.2f}' for c in CENTRES]
print(f"I5 library: {len(LIB)} sets; level {int(LIB['level set'].sum())} (up {int((LIB['level set'] & (LIB.L > 0)).sum())}), "
      f"location {int(LIB['location set'].sum())}, gradient erased {int(LIB.gradient.eq('erased').sum())}, kept {int(LIB.gradient.eq('kept').sum())}; "
      f"genome-wide survival {A_GENOME:.3f}")


READING.append('I5 post hoc: the set z of L and M recomputed for each relabeling with the same nulls; the number of '
               'sets with q ≤ 0.05 there, against the condition, is the calibration of the random-set null (it ignores '
               'correlation between genes within a set).')


def relabel_rows(k):
    Lk, Pk = DEC[k][0], DEC[k][1]

    def one(s, m):
        _, zl, pl = pdc.set_mean_z(Lk, m, PERMS)
        _, _, zm, pm = pdc.set_profile_z(Pk, m, PERMS)
        return {'set': s, 'z L': zl, 'p L': pl, 'z M': zm, 'p M': pm}

    def run():
        return pd.DataFrame(Parallel(n_jobs=N_JOBS, prefer='threads')(delayed(one)(s, m) for s, m in LIB_M.items()))
    return cached_table(f'i5_relabel_{k}', run, params={'n_perm': N_PERM, 'seed': SEED},
                        inputs={'L': Lk, 'P': Pk, 'perms': PERMS, 'sets': LIB_M})


CALIB = {}
setL = {'condition': LIB.L.to_numpy(), 'R1': LIB['L R1'].to_numpy(), 'R2': LIB['L R2'].to_numpy()}
setM = {'condition': LIB.M.to_numpy(), 'R1': LIB['M R1'].to_numpy(), 'R2': LIB['M R2'].to_numpy()}
for k in ('R1', 'R2'):
    t = relabel_rows(k).set_index('set').loc[LIB.set]
    qL = multipletests(t['p L'], method='fdr_bh')[1]
    qM = multipletests(t['p M'], method='fdr_bh')[1]
    others = [o for o in setL if o != k]
    CALIB[k] = {'median z L': float(t['z L'].median()), 'median z M': float(t['z M'].median()),
                'sets q L ≤ 0.05': int((qL <= Q).sum()), 'sets q M ≤ 0.05': int((qM <= Q).sum()),
                'level rule (no pair clause)': int(((qL <= Q) & np.all([np.abs(setL[o]) <= RELABEL * np.abs(setL[k]) for o in others], axis=0)).sum()),
                'location rule (no replication clause)': int(((qM <= Q) & np.all([setM[o] <= RELABEL * setM[k] for o in others], axis=0)).sum())}
CALIB['condition'] = {'median z L': float(LIB['z L'].median()), 'median z M': float(LIB['z M'].median()),
                      'sets q L ≤ 0.05': int((LIB['q L'] <= Q).sum()), 'sets q M ≤ 0.05': int((LIB['q M'] <= Q).sum()),
                      'level rule (no pair clause)': int(((LIB['q L'] <= Q) & (LIB['L R1'].abs() <= RELABEL * LIB.L.abs()) & (LIB['L R2'].abs() <= RELABEL * LIB.L.abs())).sum()),
                      'location rule (no replication clause)': int(((LIB['q M'] <= Q) & (LIB['M R1'] <= RELABEL * LIB.M) & (LIB['M R2'] <= RELABEL * LIB.M)).sum())}
CALIB = pd.DataFrame(CALIB)
display(CALIB.round(2))


def top(t, mask, key, n=12, ascending=False):
    s = t[mask].sort_values(key, ascending=ascending)
    return s.drop_duplicates('group') if 'group' in s else s


SHOW = ['name', 'members', 'L', 'z L', 'q L', 'M', 'z M', 'q M', 'peak', 'r IR mice', 'A', 'z A', 'q A', 'A R1', 'A R2',
        'FC DCT', 'FC Thin_Limb']
for title, mask, key, asc in (('level up', LIB['level set'] & (LIB.L > 0), 'z L', False),
                              ('level down', LIB['level set'] & (LIB.L < 0), 'z L', True),
                              ('location', LIB['location set'], 'z M', False),
                              ('gradient erased', LIB.gradient.eq('erased'), 'z A', True),
                              ('gradient kept', LIB.gradient.eq('kept'), 'z A', False)):
    print(f'— {title}')
    display(top(LIB, mask, key, ascending=asc)[SHOW].head(15).round(3))
display(EXT[['set', 'members', 'L', 'z L', 'q L', 'level set', 'M', 'z M', 'q M', 'location set', 'peak', 'A', 'z A',
             'q A', 'gradient', 'FC DCT', 'FC Thin_Limb']].round(3))

SIG = {'time of day (ZT18 − ZT6)': TOD.lfc, 'fasting (fasted − fed)': FAST.lfc, 'sex (female − male)': SEX,
       'DCT (IR − control)': pd.Series(FC_NONPT['DCT'], index=GENES),
       'thin limb (IR − control)': pd.Series(FC_NONPT['Thin_Limb'], index=GENES)}
AGREE = pd.DataFrame({k: {sname: float(spearmanr(DEC[k][0], s.reindex(GENES).to_numpy(), nan_policy='omit')[0])
                          for sname, s in SIG.items()} for k in ('condition', 'R1', 'R2')})
display(AGREE.round(3))

## 8 · Figure 91a · Injury decomposition

In [ ]:
fig = plt.figure(figsize=(183 * MM, 150 * MM), layout='constrained')
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 1.25])
ax = fig.add_subplot(gs[0, 0])
ax.scatter(G.L, G.M, s=1.5, color=LIGHT, lw=0, rasterized=True)
for cname, col in CALL_COLOR.items():
    s = G[G['call'].eq(cname)]
    ax.scatter(s.L, s.M, s=2.5, color=col, lw=0, label=f'{cname} ({len(s):,})', rasterized=True)
ax.axhline(LOCATION, color=GREY, lw=.5, ls=':')
for v in (-LEVEL, LEVEL):
    ax.axvline(v, color=GREY, lw=.5, ls=':')
ax.set_xlabel('level L (log2, IR − control)')
ax.set_ylabel('location size M (log2)')
ax.set_xlim(np.nanquantile(G.L, [.0005, .9995]))
ax.set_ylim(0, np.nanquantile(G.M, .9995))
ax.legend(loc='upper left', markerscale=3, handletextpad=.2)
panel(ax, 'a')

ax = fig.add_subplot(gs[0, 1])
bins_a = np.linspace(-1, 2, 61)
for k, col in (('condition', '#B2182B'), ('R1', GREY), ('R2', INK)):
    ax.hist(np.clip(G.loc[ZONATED, f'A {k}'], -1, 2), bins=bins_a, histtype='step', color=col, lw=.9,
            label=f"{'IR vs control' if k == 'condition' else k} ({SURV_TAB.loc['whole PT', k]:.2f})")
for v in (0, 1):
    ax.axvline(v, color=LIGHT, lw=.6, zorder=0)
ax.set_xlabel(f'gradient survival A ({int(ZONATED.sum()):,} zonated genes)')
ax.set_ylabel('genes')
ax.legend(loc='upper right', handlelength=1)
panel(ax, 'b')

ax = fig.add_subplot(gs[1, 0])
rows_s = ['whole PT'] + [r for r in SURV_TAB.index if r.startswith('S1→S2') or r.startswith('S2→S3')]
xx = np.arange(len(rows_s))
for j, (k, col) in enumerate((('condition', '#B2182B'), ('R1', GREY), ('R2', INK))):
    ax.bar(xx + (j - 1) * .26, SURV_TAB.loc[rows_s, k].astype(float), width=.25, color=col,
           label='IR vs control' if k == 'condition' else k)
ax.axhline(1, color=LIGHT, lw=.6, zorder=0)
ax.set_xticks(xx, [r.split(' (')[0] for r in rows_s])
ax.set_ylabel('pooled gradient survival')
ax.legend(loc='upper center', handlelength=1, ncol=3, bbox_to_anchor=(.5, 1.08))
panel(ax, 'c')

ax = fig.add_subplot(gs[1, 1])
lw_ = G[LEVEL_CALL]
cols = lw_['kidney-wide'].map({'kidney-wide': '#009E73', 'PT-only': '#D55E00', 'mixed': GREY})
ax.scatter(lw_.L, (lw_['FC DCT'] + lw_['FC Thin_Limb']) / 2, s=3, c=cols, lw=0, rasterized=True)
lim = np.nanquantile(np.abs(np.concatenate([lw_.L, lw_['FC DCT']])), .995)
ax.plot([-lim, lim], [-lim, lim], color=INK, lw=.4, ls=':')
ax.axhline(0, color=LIGHT, lw=.6, zorder=0)
ax.axvline(0, color=LIGHT, lw=.6, zorder=0)
for lab, col in (('kidney-wide', '#009E73'), ('PT-only', '#D55E00'), ('mixed', GREY)):
    ax.scatter([], [], s=6, color=col, label=f"{lab} ({int(lw_['kidney-wide'].eq(lab).sum())})")
ax.set_xlabel('PT level L (level-change genes)')
ax.set_ylabel('mean DCT and thin-limb log2 FC')
ax.legend(loc='upper left', handletextpad=.2)
panel(ax, 'd')

ax = fig.add_subplot(gs[:, 2])
blocks = [('level up', LIB['level set'] & (LIB.L > 0), 'z L', False, '#0072B2'),
          ('level down', LIB['level set'] & (LIB.L < 0), 'z L', True, '#56B4E9'),
          ('location', LIB['location set'], 'z M', False, '#D55E00'),
          ('gradient erased', LIB.gradient.eq('erased'), 'z A', True, '#B2182B'),
          ('gradient kept', LIB.gradient.eq('kept'), 'z A', False, '#2166AC')]
labels_, vals, colors_ = [], [], []
for title, mask, key, asc, col in blocks:
    s = top(LIB, mask, key, ascending=asc).head(6)
    labels_ += list(s.name)
    vals += list(s[key])
    colors_ += [col] * len(s)
    labels_.append('')
    vals.append(np.nan)
    colors_.append('none')
y = np.arange(len(labels_))[::-1]
ax.barh(y, vals, color=colors_, height=.75)
ax.set_yticks(y, labels_, fontsize=5)
ax.axvline(0, color=INK, lw=.4)
ax.set_xlabel('set z (against expression-matched random sets)')
ax.legend(handles=[Patch(color=col, label=title) for title, _, _, _, col in blocks], loc='lower right', fontsize=5.5,
          handlelength=1)
panel(ax, 'e')
save(fig, 'fig91a_injury_decomposition')

## 9 · Figure 91b · Where injury acts, and what it switches on and off

In [ ]:
fig = plt.figure(figsize=(183 * MM, 150 * MM), layout='constrained')
gs = fig.add_gridspec(2, 3)
ax = fig.add_subplot(gs[0, 0])
nf = SHAPE_NULL.loc[CLASS_ORDER]
xx = np.arange(len(CLASS_ORDER))
for j, (k, col) in enumerate((('condition', '#B2182B'), ('R1', GREY), ('R2', INK))):
    ax.bar(xx + (j - 1) * .26, nf[k], width=.25, color=col, label='IR vs control' if k == 'condition' else k)
ax.set_xticks(xx, CLASS_ORDER, rotation=30, ha='right')
ax.set_ylabel('genes (injury shape class)')
ax.legend(loc='upper right', handlelength=1)
panel(ax, 'a')

ax = fig.add_subplot(gs[0, 1])
loc_order = ['inside S1', 'S1/S2', 'inside S2', 'S2/S3', 'inside S3']
sl = SHAPE_LOC.reindex(columns=[c for c in loc_order if c in SHAPE_LOC.columns]).fillna(0)
bottom = np.zeros(sl.shape[1])
for cls in [c for c in CLASS_ORDER if c in sl.index]:
    ax.bar(np.arange(sl.shape[1]), sl.loc[cls], bottom=bottom, color=CLASS_COLOR[cls], label=cls, width=.7)
    bottom += sl.loc[cls].to_numpy()
ax.set_xticks(np.arange(sl.shape[1]), sl.columns, rotation=30, ha='right')
ax.set_ylabel('location-change genes')
ax.legend(loc='upper left', fontsize=5.5, handlelength=1)
panel(ax, 'b')

ax = fig.add_subplot(gs[0, 2])
for n in AKI4:
    ax.plot(CENTRES, PROG[('induced', 'all')][n], color=SPEC_COLOR[n], lw=1, label=n)
    ax.plot(CENTRES, PROG[('induced', 'on curve')][n], color=SPEC_COLOR[n], lw=.6, ls=':')
seg_lines(ax)
ax.set_xlabel('PT position (segment units)')
ax.set_ylabel(f'induced program: share of {len(IND)} genes detected')
ax.legend(loc='upper left', handlelength=1)
panel(ax, 'c')

ax = fig.add_subplot(gs[1, 0])
for n in AKI4:
    ax.plot(CENTRES, PROG[('lost', 'all')][n], color=SPEC_COLOR[n], lw=1, label=n)
seg_lines(ax)
ax.set_xlabel('PT position (segment units)')
ax.set_ylabel(f'lost program: share of {len(LOS)} genes detected')
panel(ax, 'd')

ax = fig.add_subplot(gs[1, 1])
im = ax.imshow(TOP_DET, aspect='auto', cmap='magma', vmin=0, vmax=1,
               extent=(CENTRES[0] - .05, CENTRES[-1] + .05, len(TOP_IND) - .5, -.5))
ax.set_yticks(np.arange(len(TOP_IND)), GENES_ALL[TOP_IND], fontsize=4.5)
seg_lines(ax)
ax.set_xlabel('PT position (segment units)')
cb = fig.colorbar(im, ax=ax, fraction=.05, pad=.02)
cb.set_label('share of IR units detecting')
panel(ax, 'e')

ax = fig.add_subplot(gs[1, 2])
locs = top(LIB, LIB['location set'], 'z M').head(20)
H = locs[PCOLS].to_numpy(float)
order = np.argsort(locs.peak.to_numpy())
lim = np.nanquantile(np.abs(H), .98) if len(H) else 1
im = ax.imshow(H[order], aspect='auto', cmap='RdBu_r', norm=TwoSlopeNorm(0, -lim, lim),
               extent=(CENTRES[0] - .05, CENTRES[-1] + .05, len(H) - .5, -.5))
ax.set_yticks(np.arange(len(H)), locs.name.to_numpy()[order], fontsize=5)
seg_lines(ax)
ax.set_xlabel('PT position (segment units)')
cb = fig.colorbar(im, ax=ax, fraction=.05, pad=.02)
cb.set_label('location profile (log2)')
panel(ax, 'f')
save(fig, 'fig91b_where_injury_acts')

## 10 · Part II · Species gradient programs

90's end contrasts E per specimen. Graded in a species: |E| ≥ 0.5 in both its specimens, same sign. Sets: the
gradient cosine C over members graded in either species, and the amplitude ratio Q = log2(RMS E_h / RMS E_m);
random sets of graded-in-either genes matched by expression stratum.

In [ ]:
D = pd.read_csv(T90 / 'd_end_contrasts.csv.gz')
A90 = pd.read_csv(T90 / 'a_gene_classes.csv.gz', usecols=['gene', 'log2 CP10k'])
E = {n: D[f'E {n}'].to_numpy(float) for n in MOUSE + HUMAN}
EM, EH = D['E mouse'].to_numpy(float), D['E human'].to_numpy(float)


def graded(a, b):
    return (np.abs(a) >= GRADED) & (np.abs(b) >= GRADED) & (np.sign(a) == np.sign(b))


GM, GHU = graded(E['Ctrl1A2'], E['Ctrl1A4']), graded(E['HUK1_COR1'], E['HUK1_MED1'])
EITHER = np.flatnonzero(GM | GHU)
with np.errstate(divide='ignore', invalid='ignore'):
    A_H = np.where(GM, EH / EM, np.nan)
    A_M = np.where(GHU, EM / EH, np.nan)
S1_COUNTS = pd.DataFrame({'mouse-graded: human survival': pd.Series(pdc.survival_class(A_H[GM])).value_counts(),
                          'human-graded: mouse survival': pd.Series(pdc.survival_class(A_M[GHU])).value_counts()}) \
    .reindex(list(pdc.SURVIVAL_CLASSES)).fillna(0).astype(int)
display(S1_COUNTS)
cp = D[['mouse']].merge(A90, left_on='mouse', right_on='gene', how='left')['log2 CP10k'].to_numpy(float)[EITHER]
STRATA_S = np.full(len(EITHER), -1)
okc = np.isfinite(cp)
STRATA_S[okc] = pd.qcut(cp[okc], 10, labels=False, duplicates='drop')
PERMS_S = pp.strata_permutations(STRATA_S, N_PERM, np.random.default_rng(SEED + 1))
eidx = pd.Series(np.arange(len(EITHER)), index=D.mouse.to_numpy()[EITHER])
SETS_H = json.loads((T71 / 'gene_sets_human_vs_mouse.json').read_text())
SP_M = {}
for s, g in SETS_H.items():
    m = np.sort(eidx.reindex(list(dict.fromkeys(g))).dropna().astype(int).to_numpy())
    if len(m) >= MIN_SET:
        SP_M[s] = m
em, eh = EM[EITHER], EH[EITHER]
es = {n: E[n][EITHER] for n in E}


def species_row(s, m):
    out = {'set': s, 'members': len(m)}
    out['C'], out['z C'], out['p C'] = pdc.set_cosine_z(em, eh, m, PERMS_S)
    out['Q'], out['z Q'], out['p Q'] = pdc.set_log_rms_ratio_z(em, eh, m, PERMS_S)
    for a in MOUSE:
        for b in HUMAN:
            out[f'C {a}×{b}'] = pdc.gradient_cosine(es[a][m], es[b][m])
    out['C mouse A2×A4'] = pdc.gradient_cosine(es['Ctrl1A2'][m], es['Ctrl1A4'][m])
    out['C human COR1×MED1'] = pdc.gradient_cosine(es['HUK1_COR1'][m], es['HUK1_MED1'][m])
    out['E mouse'], out['E human'] = float(em[m].mean()), float(eh[m].mean())
    return out


def species_compute():
    return pd.DataFrame(Parallel(n_jobs=N_JOBS, prefer='threads')(delayed(species_row)(s, m) for s, m in SP_M.items()))


SP = cached_table('s2_species_sets', species_compute, params={'n_perm': N_PERM, 'seed': SEED + 1},
                  inputs={'em': em, 'eh': eh, 'es': [es[n] for n in E], 'perms': PERMS_S, 'sets': SP_M})
for s in ('C', 'Q'):
    SP[f'q {s}'] = multipletests(SP[f'p {s}'], method='fdr_bh')[1]
pairc = SP[[f'C {a}×{b}' for a in MOUSE for b in HUMAN]]
within = (SP['C mouse A2×A4'] >= .5) & (SP['C human COR1×MED1'] >= .5)
SP['reversed'] = (SP.C < 0) & (SP['q C'] <= Q) & (pairc < 0).all(axis=1) & within
SP['conserved'] = (SP['z C'] > 0) & (SP['q C'] <= Q) & (pairc > 0).all(axis=1) & within
SP['dominant'] = np.where((SP['q Q'] <= Q) & (SP.Q < 0) & (SP['C mouse A2×A4'] >= .5), 'mouse',
                          np.where((SP['q Q'] <= Q) & (SP.Q > 0) & (SP['C human COR1×MED1'] >= .5), 'human', ''))
SP['name'] = [short(s) for s in SP.set]
SP['group'] = SP.set.map(pp.redundancy_groups({s: set(SP_M[s]) for s in SP.set}))
C_GENOME = pdc.gradient_cosine(em, eh)
print(f"S2: {len(SP)} sets; reversed {int(SP.reversed.sum())}, conserved {int(SP.conserved.sum())}, mouse-dominant "
      f"{int(SP.dominant.eq('mouse').sum())}, human-dominant {int(SP.dominant.eq('human').sum())}; genome cosine {C_GENOME:.3f}")
SPSHOW = ['name', 'members', 'C', 'z C', 'q C', 'Q', 'z Q', 'q Q', 'E mouse', 'E human', 'C mouse A2×A4', 'C human COR1×MED1']
for title, mask, key, asc in (('reversed', SP.reversed, 'z C', True), ('conserved', SP.conserved, 'z C', False),
                              ('mouse-dominant', SP.dominant.eq('mouse'), 'z Q', True),
                              ('human-dominant', SP.dominant.eq('human'), 'z Q', False)):
    print(f'— {title}')
    display(SP[mask].sort_values(key, ascending=asc).drop_duplicates('group')[SPSHOW].head(15).round(3))

### S3 · Does the human gradient carry the mouse injury pattern? (repeats a scratch check)

In [ ]:
dl = pd.Series(G['Δ late'].to_numpy(), index=GENES)
LINK = []
for k in ('condition', 'R1', 'R2'):
    lk = pd.Series(np.nanmean(DELTA[k][:, VALID & (CENTRES >= LATE)], axis=1), index=GENES)
    x = lk.reindex(D.mouse).to_numpy()
    ok = np.isfinite(x) & np.isfinite(EM) & np.isfinite(EH)
    Xd = np.column_stack([np.ones(ok.sum()), EM[ok], x[ok]])
    beta = np.linalg.lstsq(Xd, EH[ok], rcond=None)[0]
    res = EH[ok] - Xd[:, :2] @ np.linalg.lstsq(Xd[:, :2], EH[ok], rcond=None)[0]
    LINK.append({'Δ late from': k, 'genes': int(ok.sum()), 'β E mouse': beta[1], 'β Δ late': beta[2],
                 'Spearman(residual, Δ late)': float(spearmanr(res, x[ok])[0])})
LINK = pd.DataFrame(LINK)
display(LINK.round(3))
both_sets = SP.merge(LIB[['set', 'A', 'z A']], on='set').dropna(subset=['A'])
LINK_SET = float(spearmanr(both_sets.C, both_sets.A)[0])
print(f'S3 set level: Spearman(species cosine, injury gradient survival) over {len(both_sets)} sets = {LINK_SET:.3f}')

## 11 · Figure 91c · Species gradient programs

In [ ]:
fig = plt.figure(figsize=(183 * MM, 150 * MM), layout='constrained')
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 1.3])
ax = fig.add_subplot(gs[0, 0])
cat_color = {'conserved': '#009E73', 'reversed': '#B2182B', 'mouse only': '#0072B2', 'human only': '#D55E00'}
ax.scatter(EM, EH, s=1.5, color=LIGHT, lw=0, rasterized=True)
for c, col in cat_color.items():
    s = D.category.eq(c).to_numpy()
    ax.scatter(EM[s], EH[s], s=3, color=col, lw=0, label=f'{c} ({int(s.sum())})', rasterized=True)
ax.axhline(0, color=GREY, lw=.4)
ax.axvline(0, color=GREY, lw=.4)
ax.set_xlabel('mouse end contrast E (late − early, log2)')
ax.set_ylabel('human end contrast E')
ax.legend(loc='upper left', markerscale=3, handletextpad=.2, fontsize=5.5)
panel(ax, 'a')

ax = fig.add_subplot(gs[0, 1])
ax.scatter(SP['C mouse A2×A4'].clip(-1, 1), SP.C, s=3, color=GREY, lw=0)
for col, mask, lab in (('#B2182B', SP.reversed, 'reversed'), ('#009E73', SP.conserved, 'conserved')):
    ax.scatter(SP.loc[mask, 'C mouse A2×A4'], SP.loc[mask, 'C'], s=6, color=col, lw=0, label=f'{lab} ({int(mask.sum())})')
ax.axhline(C_GENOME, color=INK, lw=.5, ls=':')
ax.axhline(0, color=LIGHT, lw=.6, zorder=0)
ax.set_xlabel('set cosine, mouse A2 × A4')
ax.set_ylabel('set cosine, mouse × human')
ax.legend(loc='lower left', handletextpad=.2)
panel(ax, 'b')

ax = fig.add_subplot(gs[:, 2])
# Ranked lists whether or not a set is called; a called set is coloured, an uncalled one grey.
blocks = [('lowest cosine z', 'C', True, SP.reversed, '#B2182B'), ('highest cosine z', 'C', False, SP.conserved, '#009E73'),
          ('lowest amplitude z (mouse-dominant)', 'Q', True, SP.dominant.eq('mouse'), '#0072B2'),
          ('highest amplitude z (human-dominant)', 'Q', False, SP.dominant.eq('human'), '#D55E00')]
labels_, vals, colors_ = [], [], []
for title, key, asc, called, col in blocks:
    s = SP.sort_values(f'z {key}', ascending=asc).drop_duplicates('group').head(6)
    labels_ += list(s.name)
    vals += list(s[f'z {key}'])
    colors_ += [col if c else GREY for c in called[s.index]]
    labels_.append('')
    vals.append(np.nan)
    colors_.append('none')
y = np.arange(len(labels_))[::-1]
ax.barh(y, vals, color=colors_, height=.75)
ax.set_yticks(y, labels_, fontsize=5)
ax.axvline(0, color=INK, lw=.4)
ax.set_xlabel('set z (cosine C or amplitude ratio Q)')
ax.legend(handles=[Patch(color=col, label=f'{t.split(" (")[0]}, called') for t, _, _, _, col in blocks]
          + [Patch(color=GREY, label='not called')], loc='lower left', fontsize=5.5, handlelength=1)
panel(ax, 'c')

ax = fig.add_subplot(gs[1, 0])
sc_ = S1_COUNTS.div(S1_COUNTS.sum())
xx = np.arange(len(sc_))
ax.bar(xx - .2, sc_.iloc[:, 0], width=.38, color='#0072B2', label='mouse-graded genes, human')
ax.bar(xx + .2, sc_.iloc[:, 1], width=.38, color='#D55E00', label='human-graded genes, mouse')
ax.set_xticks(xx, sc_.index, rotation=30, ha='right')
ax.set_ylabel('share of graded genes')
ax.legend(loc='upper left', handlelength=1, fontsize=5.5)
panel(ax, 'd')

ax = fig.add_subplot(gs[1, 1])
ax.bar(np.arange(3), LINK['Spearman(residual, Δ late)'], color=['#B2182B', GREY, INK])
ax.set_xticks(np.arange(3), ['IR vs control', 'R1', 'R2'])
ax.axhline(0, color=INK, lw=.4)
ax.set_ylabel('ρ (human gradient beyond mouse, late injury Δ)')
panel(ax, 'e')
save(fig, 'fig91c_species_gradient_programs')

## 12 · Recovery checks, readout and run record

In [ ]:
gi = G.set_index('gene')
circ = LIB[LIB.set.str.contains('Circadian Clock R-HSA-400253|Circadian rhythm', regex=True)]
tod = EXT[EXT.set.str.startswith('time of day')]
clock_genes = [g for g in ('Dbp', 'Tef', 'Hlf', 'Nr1d1', 'Nr1d2', 'Per1', 'Per2', 'Per3', 'Ciart') if g in gi.index]
hyp = SP[SP.set.str.contains('Hypoxia')]
RECOVERY = pd.DataFrame([
    {'check': '1 · Slc7a13 erased or reversed; S2→S3 survival below S1→S2',
     'value': f"Slc7a13 A {gi.loc['Slc7a13', 'A condition']:.2f} ({gi.loc['Slc7a13', 'survival']}); "
              + '; '.join(f'{r} {SURV_TAB.loc[r, "condition"]:.2f}' for r in rows_s[1:])},
    {'check': '2 · replicated interior dip of the induced program in S1',
     'value': '; '.join(f"{r.units} {r.mouse}: dip {r.dip} at {r.position:.2f}" for r in DIPS.itertuples())},
    {'check': '3 · circadian sets or a time-of-day program are level sets; clock genes kidney-wide',
     'value': '; '.join(f"{short(r.set)}: L {r.L:.2f}, q {r['q L']:.3g}, level set {r['level set']}" for _, r in pd.concat([circ, tod]).iterrows())
              + ' | ' + ', '.join(f"{g} {gi.loc[g, 'kidney-wide']}" for g in clock_genes)},
    {'check': '4 · Hallmark hypoxia reversed or mouse-dominant',
     'value': '; '.join(f"{short(r.set)}: C {r.C:.2f} (q {r['q C']:.3g}), Q {r.Q:.2f} (q {r['q Q']:.3g}), reversed {r.reversed}, dominant '{r.dominant}'"
                        for _, r in hyp.iterrows())},
])
display(RECOVERY)
READOUT = pd.DataFrame([
    {'part': 'I1', 'readout': 'gene calls', 'value': json.dumps(G['call'].value_counts().to_dict())},
    {'part': 'I1', 'readout': 'specificity (condition; R1; R2)', 'value': json.dumps(I1_SPEC)},
    {'part': 'I1', 'readout': 'injury shape classes (condition; R1; R2)', 'value': json.dumps(SHAPE_NULL.to_dict())},
    {'part': 'I2', 'readout': 'pooled survival', 'value': json.dumps(SURV_TAB.astype(float).round(4).to_dict())},
    {'part': 'I2', 'readout': 'survival classes (condition)', 'value': json.dumps(SURV_COUNTS.to_dict())},
    {'part': 'I3', 'readout': 'level-change genes by kidney-wide call', 'value': json.dumps(KW.to_dict())},
    {'part': 'I4', 'readout': 'programs (induced; lost); replicated dip',
     'value': f'{len(IND)}; {len(LOS)}; {json.dumps(DIP_REP)}'},
    {'part': 'I5', 'readout': 'library sets: level (up, down); location; erased; kept',
     'value': f"{int((LIB['level set'] & (LIB.L > 0)).sum())}, {int((LIB['level set'] & (LIB.L < 0)).sum())}; "
              f"{int(LIB['location set'].sum())}; {int(LIB.gradient.eq('erased').sum())}; {int(LIB.gradient.eq('kept').sum())}"},
    {'part': 'I5', 'readout': 'post hoc relabel calibration of set z', 'value': json.dumps(CALIB.round(3).to_dict())},
    {'part': 'I4', 'readout': 'post hoc induced-gene zones', 'value': json.dumps(IND_GENES.zone.value_counts().to_dict())},
    {'part': 'I5', 'readout': 'signature agreement (Spearman with L)', 'value': json.dumps(AGREE.round(4).to_dict())},
    {'part': 'S1', 'readout': 'gene survival classes', 'value': json.dumps(S1_COUNTS.to_dict())},
    {'part': 'S2', 'readout': 'sets: reversed; conserved; mouse-dominant; human-dominant',
     'value': f"{int(SP.reversed.sum())}; {int(SP.conserved.sum())}; {int(SP.dominant.eq('mouse').sum())}; {int(SP.dominant.eq('human').sum())}"},
    {'part': 'S3', 'readout': 'injury link (gene β, residual ρ; set ρ)',
     'value': json.dumps(LINK.set_index('Δ late from')[['β Δ late', 'Spearman(residual, Δ late)']].round(4).to_dict()) + f'; {LINK_SET:.4f}'},
])
for path, frame in (('i_genes.csv.gz', G), ('i_induced_genes.csv', IND_GENES), ('i_lost_genes.csv', LOS_GENES),
                    ('i_dips.csv', DIPS), ('i_survival.csv', SURV_TAB.reset_index()), ('i5_library_sets.csv', LIB),
                    ('i5_external_programs.csv', EXT), ('i5_signature_agreement.csv', AGREE.reset_index()),
                    ('s2_species_sets.csv', SP), ('s3_injury_link.csv', LINK), ('recovery_checks.csv', RECOVERY),
                    ('i5_relabel_calibration.csv', CALIB.reset_index()), ('readout.csv', READOUT)):
    frame.to_csv(tables / path, index=False)
pd.DataFrame({f'{p} {sens} {n}': PROG[(p, sens)][n] for (p, sens) in PROG for n in AKI4}, index=CENTRES).to_csv(tables / 'i4_program_profiles.csv')
display(READOUT)
record = {'logic': NOTEBOOK_LOGIC_VERSION, 'guards': GUARDS, 'reading': READING, 'seconds': SECONDS,
          'units': pd.Series(spec).value_counts().to_dict(), 'cuts': list(CUTS),
          'genes': {'shared': int(SHARED.sum()), 'reference': int(REF.sum()), 'induced': int(INDUCED.sum()),
                    'lost': int(LOST.sum()), 'zonated': int(ZONATED.sum())},
          'sets': {'library': len(LIB), 'external': len(EXT), 'species': len(SP)},
          'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=float))
print(f"Done in {record['wall_seconds'] / 60:.1f} min")